# Mineração de Dados — PL5 (Text Mining & NLP II)

## Setup Inicial

In [1]:
%pip install nltk spacy gensim matplotlib scikit-learn

  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached weasel-1.0.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached confection-1.3.3-py3-none-any.whl.metadata (19 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached cloudpathlib-0.23.0-py3-none-any.whl.metadata (16 kB)
   ---------------------------------------- 0.0/14.2 MB ? eta -:--:--
   ---------------------- ----------------- 7.9/14.2 MB 44.2 MB/s eta 0:00:01
   ---------------------------------------  14.2/14.2 MB 37.0 MB/s eta 0:00:01
   ---------------------------------------- 14.2/14.2 MB 33.1 MB/s  0:00:00
Using cached catalogue-2.0.10-py3-none-any.whl (17 kB)
Using cached confection-1.3.

In [2]:
import spacy
import string
import math
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

import nltk
from nltk.corpus import stopwords, wordnet as wn
from nltk.tokenize import word_tokenize
from sklearn.manifold import TSNE

import gensim.downloader as api
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

# Downloads do NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Instalação e carregamento do modelo spaCy para português
!python -m spacy download pt_core_news_sm
nlp = spacy.load("pt_core_news_sm")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LuísFigueiredo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LuísFigueiredo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\LuísFigueiredo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\LuísFigueiredo\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


     ---------------------------------------- 0.0/13.0 MB ? eta -:--:--
      --------------------------------------- 0.3/13.0 MB ? eta -:--:--
     ------------------------------- ------- 10.5/13.0 MB 50.4 MB/s eta 0:00:01
     ---------------------------------------- 13.0/13.0 MB 35.3 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')


## Parte 1 - VSM e Similaridade de Cosseno

In [3]:
documentos = [
    "O gato preto dorme no sofá da sala",
    "O cachorro late para o gato preto",
    "Gatos e cachorros são animais domésticos",
    "A sala tem um sofá confortável",
    "Animais selvagens vivem na floresta"
]

query = "gato preto no sofa"

### Questão 1.1. Implemente uma função de pré-processamento

In [4]:
stop_words = set(stopwords.words('portuguese'))
pontuacao = set(string.punctuation)

def preprocess_text(texto):
    # 1. Tokenizar o texto em minúsculas
    tokens_low = word_tokenize(texto.lower(), language='portuguese')
    
    # 2. Remover stopwords e pontuação (mantendo apenas palavras alfanuméricas para remover lixo)
    tokens_processados = [
        tok for tok in tokens_low 
        if tok not in stop_words and tok not in pontuacao and tok.isalnum()
    ]
    
    return tokens_processados

print("Teste da função:")
print(preprocess_text("O gato preto dorme no sofá!"))
print()

documentos_processados = [preprocess_text(doc) for doc in documentos]
query_processada = preprocess_text(query)

print("Documentos processados:")
for i, doc in enumerate(documentos_processados):
    print(f"Doc {i+1}: {doc}")
print(f"Query processada: {query_processada}")

Teste da função:
['gato', 'preto', 'dorme', 'sofá']

Documentos processados:
Doc 1: ['gato', 'preto', 'dorme', 'sofá', 'sala']
Doc 2: ['cachorro', 'late', 'gato', 'preto']
Doc 3: ['gatos', 'cachorros', 'animais', 'domésticos']
Doc 4: ['sala', 'sofá', 'confortável']
Doc 5: ['animais', 'selvagens', 'vivem', 'floresta']
Query processada: ['gato', 'preto', 'sofa']


### Questão 1.2. Implemente manualmente o cálculo de TF-IDF

In [5]:
def compute_tf(tokens):
    total_termos = len(tokens)
    if total_termos == 0:
        return {}
        
    contagem = Counter(tokens)
    tf_dict = {t: c / total_termos for t, c in contagem.items()}
    return tf_dict

def compute_idf(documentos_processados):
    N = len(documentos_processados)
    df = {}
    
    for doc in documentos_processados:
        termos_unicos = set(doc)
        for t in termos_unicos:
            df[t] = df.get(t, 0) + 1
            
    idf_dict = {t: math.log(N / count) for t, count in df.items()}
    return idf_dict

def compute_tfidf(doc_tokens, idf_dict):
    tf_dict = compute_tf(doc_tokens)
    tfidf_dict = {}
    
    for t, val_tf in tf_dict.items():
        if t in idf_dict:
            tfidf_dict[t] = val_tf * idf_dict[t]
            
    return tfidf_dict

# Calcular IDF
idf = compute_idf(documentos_processados)

print("IDF (termos relevantes):")
termos_mostrar = ['gato', 'preto', 'sofá', 'sala', 'animais']
for termo in termos_mostrar:
    if termo in idf:
        print(f" {termo}: {idf[termo]:.4f}")

# Calcular vetores TF-IDF
vetores_docs = [compute_tfidf(doc, idf) for doc in documentos_processados]
vetor_query = compute_tfidf(query_processada, idf)

print("\nVetor da query:")
print(vetor_query)

IDF (termos relevantes):
 gato: 0.9163
 preto: 0.9163
 sofá: 0.9163
 sala: 0.9163
 animais: 0.9163

Vetor da query:
{'gato': 0.3054302439580517, 'preto': 0.3054302439580517}


### Questão 1.3. Implemente a função de similaridade de cosseno

In [6]:
def cosine_similarity(v1, v2):
    # 1. Encontrar termos comuns
    termos_comuns = set(v1.keys()).intersection(set(v2.keys()))
    
    # 2. Calcular produto escalar
    dot_product = sum([v1[t] * v2[t] for t in termos_comuns])
    
    # 3. Calcular normas
    norm1 = math.sqrt(sum([val**2 for val in v1.values()]))
    norm2 = math.sqrt(sum([val**2 for val in v2.values()]))
    
    # 4. Evitar divisão por zero
    if norm1 == 0.0 or norm2 == 0.0:
        return 0.0
        
    # 5. Calcular similaridade
    similaridade = dot_product / (norm1 * norm2)
    return similaridade

# Calcular similaridades
print("Similaridades Query vs Documentos:")
print("-" * 50)
similaridades = []

for i, doc_vetor in enumerate(vetores_docs):
    sim = cosine_similarity(vetor_query, doc_vetor)
    similaridades.append((i, sim))
    print(f"Doc {i+1}: {sim:.4f} '{documentos[i]}'")

# Ordenar documentos por similaridade (decrescente)
similaridades_ordenadas = sorted(similaridades, key=lambda item: item[1], reverse=True)

print("\n" + "=" * 50)
print("RANKING DE RELEVÂNCIA:")
print("=" * 50)
for idx, sim in similaridades_ordenadas:
    if sim > 0:
        print(f"{sim:.4f} - Doc {idx+1}: {documentos[idx]}")

Similaridades Query vs Documentos:
--------------------------------------------------
Doc 1: 0.5313 'O gato preto dorme no sofá da sala'
Doc 2: 0.4948 'O cachorro late para o gato preto'
Doc 3: 0.0000 'Gatos e cachorros são animais domésticos'
Doc 4: 0.0000 'A sala tem um sofá confortável'
Doc 5: 0.0000 'Animais selvagens vivem na floresta'

RANKING DE RELEVÂNCIA:
0.5313 - Doc 1: O gato preto dorme no sofá da sala
0.4948 - Doc 2: O cachorro late para o gato preto


### Questão 1.4. Como é que a remoção de stopwords afetou os resultados?

A remoção de *stopwords* influencia diretamente os resultados pois descarta palavras estruturais extremamente comuns mas desprovidas de valor semântico, como "a", "de", "com". Esta filtragem potencia a etapa de Feature Extraction e permite dar o peso adequado às palavras informativas. Sem esta limpeza, os documentos poderiam obter valores de similaridade artificialmente altos só pela partilha de conectores ou preposições idênticas.

### Questão 1.5. Identifique duas limitações deste modelo.

1. **Incapacidade de captar Semântica e Sinónimos:** Uma vez que este modelo de VSM faz a comparação puramente ao nível dos vocábulos exatos, não existe a perceção de que duas palavras diferentes possam partilhar o mesmo significado. A comparação de "carro" e "veículo" resultará numa similaridade nula pois os vetores são estritamente ortogonais.
2. **Problema de Ordem (Limitações do Bag-of-Words):** Ao mapear a frequência isolada de termos, ignora-se a ordem em que as palavras surgem. Isso faz com que frases de sentido oposto como "Dog bites man" e "Man bites dog" apresentem uma similaridade de 100% no cálculo de cosseno.

## Parte 2 - WordNet - Explorando Relações Semânticas

### Questão 2.1. Explore os synsets de uma palavra

In [7]:
def explorar_palavra(palavra, lang='por'):
    # Obter synsets
    synsets = wn.synsets(palavra, lang=lang)
    
    if not synsets:
        print(f"Palavra '{palavra}' não encontrada no WordNet")
        return
        
    print(f"\n{'='*50}")
    print(f"Explorando: {palavra}")
    print(f"Número de synsets: {len(synsets)}")
    
    for i, syn in enumerate(synsets[:3]):
        print(f"\n--- Synset {i+1}: {syn.name()} ---")
        
        # Mostrar definição
        print(f"Definição: {syn.definition()}")
        
        # Mostrar exemplos (se houver)
        exemplos = syn.examples()
        if exemplos:
            print(f"Exemplos: {exemplos}")
            
        # Mostrar Lemmas (sinónimos)
        lemmas = [lemma.name() for lemma in syn.lemmas()]
        print(f"Lemmas: {lemmas}")
        
        # Mostrar hiperónimos (se houver)
        hiper = syn.hypernyms()
        if hiper:
            print(f"Hiperónimos: {[h.name() for h in hiper]}")
            
        # Mostrar hipónimos (se houver)
        hipo = syn.hyponyms()
        if hipo:
            print(f"Hipónimos: {[h.name() for h in hipo[:5]]}")

# Testar com palavras
palavras_teste = ['cachorro', 'computador', 'casa', 'amor', 'correr']
for palavra in palavras_teste:
    explorar_palavra(palavra)


Explorando: cachorro
Número de synsets: 2

--- Synset 1: bitch.n.04 ---
Definição: female of any member of the dog family
Lemmas: ['bitch']
Hiperónimos: ['canine.n.02']
Hipónimos: ['brood_bitch.n.01']

--- Synset 2: dog.n.01 ---
Definição: a member of the genus Canis (probably descended from the common wolf) that has been domesticated by man since prehistoric times; occurs in many breeds
Exemplos: ['the dog barked all night']
Lemmas: ['dog', 'domestic_dog', 'Canis_familiaris']
Hiperónimos: ['domestic_animal.n.01', 'canine.n.02']
Hipónimos: ['hunting_dog.n.01', 'poodle.n.01', 'leonberg.n.01', 'newfoundland.n.01', 'pug.n.01']

Explorando: computador
Número de synsets: 3

--- Synset 1: calculator.n.02 ---
Definição: a small machine that is used for mathematical calculations
Lemmas: ['calculator', 'calculating_machine']
Hiperónimos: ['machine.n.01']
Hipónimos: ['abacus.n.02', 'counter.n.03', 'adder.n.02', 'quipu.n.01', 'hand_calculator.n.01']

--- Synset 2: computer.n.01 ---
Definição: a 

### Questão 2.2. Similaridade entre palavras usando path similarity

In [8]:
def similaridade_wordnet(palavra1, palavra2, lang='por'):
    # Obter synsets
    syns1 = wn.synsets(palavra1, lang=lang)
    syns2 = wn.synsets(palavra2, lang=lang)
    
    if not syns1 or not syns2:
        return None, None
        
    max_sim = 0.0
    melhor_par = None
    
    # Comparar todos os pares
    for s1 in syns1[:3]:
        for s2 in syns2[:3]:
            try:
                # Calcular similaridade
                sim = s1.path_similarity(s2)
                
                # Atualizar máximo se necessário
                if sim is not None and sim > max_sim:
                    max_sim = sim
                    melhor_par = (s1, s2)
            except Exception:
                continue
                
    return max_sim, melhor_par

# Testar pares de palavras
pares = [
    ('cachorro', 'cão'),
    ('cachorro', 'gato'),
    ('cachorro', 'carro'),
    ('casa', 'residência'),
    ('casa', 'apartamento'),
    ('amor', 'ódio')
]

print("SIMILARIDADE SEMÂNTICA (WordNet)")
print("=" * 50)
for p1, p2 in pares:
    sim, par = similaridade_wordnet(p1, p2)
    if sim is not None:
        print(f"{p1} vs {p2}: {sim:.4f}")
    else:
        print(f"{p1} vs {p2}: não foi possível calcular")

SIMILARIDADE SEMÂNTICA (WordNet)
cachorro vs cão: 1.0000
cachorro vs gato: 0.2500
cachorro vs carro: 0.0909
casa vs residência: 0.3333
casa vs apartamento: 1.0000
amor vs ódio: 0.1250


## Parte 3 - Dependency Parsing com spaCy

### Questão 3.1. Analisar frases e dependências

In [9]:
def analisar_frase(frase):
    # Processar frase com spacy
    doc = nlp(frase)
    
    print(f"\nFrase: '{frase}'")
    print("-" * 60)
    print(f"{'Token':<15} {'POS':<10} {'Dependência':<15} {'Governador':<15}")
    print("-" * 60)
    
    # Iterar sobre tokens e mostrar informações
    for token in doc:
        print(f"{token.text:<15} {token.pos_:<10} {token.dep_:<15} {token.head.text:<15}")
        
    return doc

# Testar com frases
frases_teste = [
    "O cachorro mordeu o homem",
    "A menina de vestido azul comeu um bolo delicioso"
]

for frase in frases_teste:
    analisar_frase(frase)


Frase: 'O cachorro mordeu o homem'
------------------------------------------------------------
Token           POS        Dependência     Governador     
------------------------------------------------------------
O               DET        det             cachorro       
cachorro        NOUN       nsubj           mordeu         
mordeu          VERB       ROOT            mordeu         
o               DET        det             homem          
homem           NOUN       obj             mordeu         

Frase: 'A menina de vestido azul comeu um bolo delicioso'
------------------------------------------------------------
Token           POS        Dependência     Governador     
------------------------------------------------------------
A               DET        det             menina         
menina          NOUN       nsubj           comeu          
de              ADP        case            vestido        
vestido         NOUN       nmod            menina         
azul        

### Questão 3.2. Extrair relações sujeito-verbo-objeto e modificadores

In [10]:
def extrair_relacoes(doc):
    relacoes = []
    
    # Encontrar verbos e as suas relações
    for token in doc:
        if token.pos_ == "VERB":
            verbo = token.text
            
            # Encontrar sujeitos
            sujeitos = [f.text for f in token.children if f.dep_ in ["nsubj", "nsubj:pass"]]
            
            # Encontrar objetos
            objetos = [f.text for f in token.children if f.dep_ in ["obj", "iobj", "obl"]]
            
            # Criar pares sujeito-verbo-objeto
            for suj in sujeitos:
                for obj in objetos:
                    relacao = {
                        'sujeito': suj,
                        'verbo': verbo,
                        'objeto': obj
                    }
                    relacoes.append(relacao)
                    print(f" {suj} → {verbo} → {obj}")
                    
    return relacoes

def extrair_modificadores(doc):
    modificadores = []
    
    # Encontrar adjetivos e o substantivo que modificam
    for token in doc:
        if token.pos_ == "ADJ":
            # Verificar se a governador (head) é um substantivo
            substantivo = token.head
            if substantivo.pos_ == "NOUN":
                modificador = {
                    'substantivo': substantivo.text,
                    'adjetivo': token.text
                }
                modificadores.append(modificador)
                print(f" {substantivo.text} é {token.text}")
                
    return modificadores

# Teste
frase = "O carro vermelho e rápido ultrapassou o caminhão lento"
doc = nlp(frase)

print("Relações Sujeito-Verbo-Objeto:")
relacoes = extrair_relacoes(doc)

print("\nModificadores:")
modificadores = extrair_modificadores(doc)

Relações Sujeito-Verbo-Objeto:
 carro → ultrapassou → caminhão

Modificadores:
 carro é vermelho
 caminhão é lento
